<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/hf-sentiment-bert-wip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HF Transformers - Sentiment Analysis - IMDB 🎬✨

In this notebook, we'll finetune a pretrained language model (e.g., [BERT](https://huggingface.co/google-bert/bert-base-uncased)) 📚 on the [stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb) dataset 🎬 using [Hugging Face Transformers](https://huggingface.co/docs/transformers/index) 🚀.

## Setup ⚙️

First, let's install the necessary packages: 📦✨

In [ ]:
!pip install transformers datasets evaluate accelerate huggingface_hub

Define the seed for reproducibility in this notebook: 🌱📓

In [ ]:
SEED = 42

Ensure you are logged in to Hugging Face (HF_TOKEN in secrets) 🔑✨:

In [ ]:
from huggingface_hub import whoami
whoami()

Now, let's retrieve the identifier of the best training device (CPU 🖥️ vs. GPU 🎮):

In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_SUPPORTS_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f"Using device: {DEVICE}")

Let's select a base model to finetune. 🛠️ This process leverages the knowledge gained during pretraining, requiring significantly less training time ⏳ than training a model from scratch. We'll load the model later in the notebook. 📓

In [ ]:
MODEL_ID = "distilbert/distilbert-base-uncased" # 92% test set accuracy
#MODEL_ID = "bert-base-uncased" # 92% test set accuracy
#MODEL_ID = "bert-base-cased" # 92% test set accuracy
#MODEL_ID = "openai-community/gpt2" # 94% test set accuracy, starting accuracy is 50% but this is most likely due to the new classificaiton head, the better performance we get on GPT2 is probably related to the model already being able to do sentiment classification through text generation

Let's define the identifier for the [IMDB dataset](https://huggingface.co/datasets/stanfordnlp/imdb) 🎬, which pairs reviews 📝 with binary sentiment classifications (positive 👍 or negative 👎):

In [ ]:
DATASET_ID = "stanfordnlp/imdb"

Create a model card 📄 and upload it to the hub 🌐:

In [ ]:
from huggingface_hub import ModelCard, ModelCardData

# Create model card
# TODO: softcode this
HF_MODEL_ID = f"distilbert_imdb_finetune" #"{MODEL_ID}_{DATASET_ID}_finetune"
model_card = ModelCard.from_template(
    card_data=ModelCardData(
        language="en",
        license="mit",
        tags=["sentiment-analysis", MODEL_ID],
        library_name="transformers",
        model_name=HF_MODEL_ID,
        datasets=[DATASET_ID],
        metrics=["accuracy"],
    ),
    model_description=f"This model fine-tunes {MODEL_ID} on the {DATASET_ID} dataset.",
)

# Push model card to hub
model_card.push_to_hub(f"tsilva/{HF_MODEL_ID}")

## Prepare Dataset 📊✨

Load the dataset 📊:

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_ID)
raw_dataset

The dataset includes a `train` split with 25k samples 📊, a `test` split with 25k samples 🧪, and an `unsupervised` split with 50k entries 📈. Its purpose is unclear 🤔, so let's inspect it 🔍:

In [ ]:
raw_dataset["unsupervised"][0]

This is likely an unlabeled split, but let's confirm that the entire split has a label of `-1` 🔍:

In [ ]:
list(set([x["label"] for x in raw_dataset["unsupervised"]]))

`unsupervised` indicates an unsupervised split without labels. While it won't aid sentiment analysis, it may be useful for generating similar reviews ✍️ (text generation task).

Let's explore the dataset features: 📊✨

In [ ]:
raw_dataset["train"].features

Inspect samples from the `train` split: 🔍📊

In [ ]:
raw_dataset["train"][:5]["text"]

Check the variability of text lengths 📏📊:

In [ ]:
raw_dataset = raw_dataset.map(
    lambda batch: {"text_length": [len(text.split()) for text in batch["text"]]},
    batched=True
)
raw_dataset

Let's plot the distribution of text lengths 📊✍️.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram
text_lengths = [x["text_length"] for x in raw_dataset["train"]]
plt.hist(text_lengths, bins=50)

# Print stats and show histogram plot
dict(
    min=min(text_lengths),
    max=max(text_lengths),
    avg=sum(text_lengths) / len(text_lengths)
), plt

Let's extract a validation set from the training data 📊. We want to keep the test set untouched until the end 🚫 to avoid influencing our training 🏋️‍♂️.

In [ ]:
split_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=42) # Extract 10%
split_dataset["validation"] = split_dataset.pop("test")
split_dataset["test"] = raw_dataset["test"]
split_dataset

Now, let's tokenize the dataset: 🗂️✨

In [ ]:
import multiprocessing
from transformers import AutoTokenizer

# Load the tokenizer for our model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Tokenize the dataset
tokenized_dataset = split_dataset.map(
    lambda batch: tokenizer(batch["text"], truncation=True), # Define how samples are tokenized (truncate them to max model sequence length)
    batched=True, # Process in batches to minimize task overhead (eg: function calling, memory allocation, etc.)
    num_proc=multiprocessing.cpu_count(), # Tokenize in parallel using all available cores
    remove_columns=["text", "text_length"] # Drop columns we won't need, to make the dataset smaller (less memory usage)
)
tokenized_dataset

Let's instantiate a data collator for our dataset. 📊 Instead of padding the tokenized text to the maximum sequence length of the entire dataset, which is large and would add irrelevant tokens, ❌ we can use a DataCollatorWithPadding object. 🛠️ This will dynamically pad batches based on the maximum sequence length within each batch. 📏

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

# Instantiate the data collator (it will be responsible for padding
# sequence batches before running them through models)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Finetune the model 🎯✨

Load the model: 📦🔄

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load the model (architecture + weights) and adapt it for sequence classification
# (its head is swapped with a classification head for this task)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

# In case the model has no padding token,
# add one (eg: `openai-community/gpt2` doesn't have one)
if tokenizer.pad_token_id is None:
   tokenizer.pad_token_id = tokenizer.eos_token_id
   model.config.pad_token_id = tokenizer.pad_token_id

If you're using an uncased model, you might wonder how it handles cased sequences (e.g., whether to lowercase the dataset). 🤔 Run the code below to verify that the tokenizer treats the string as lowercase, eliminating the need for preprocessing: 🛠️

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer("This is a Test!")["input_ids"])

Set up the training run: 🏃‍♂️💪

In [ ]:
import torch
import numpy as np
import evaluate
from transformers import Trainer
from transformers import TrainingArguments
from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    metrics = metric.compute(predictions=predictions, references=labels)
    return metrics

training_args = TrainingArguments(
    # Where to output the model to
    output_dir=HF_MODEL_ID,
    # Identifier for model in HF hub
    hub_model_id=HF_MODEL_ID,
    # Ensure seed consistency in training arguments
    seed=SEED,
    # When train() is run again, restore from previous checkpoint
    resume_from_checkpoint=True,
    # Train for N epochs
    num_train_epochs=10,
    # Number of training samples sent to the GPU per batch.
    # Larger batches:
    #   - Increase memory usage (may cause OOM errors if too large).
    #   - Provide more stable gradients (less variance), leading to smoother updates.
    #   - Can improve training speed (fewer updates per epoch).
    #   - May lead to worse generalization (risk of converging to sharp local minima).
    # Smaller batches:
    #   - Use less memory, allowing training on limited hardware.
    #   - Introduce more gradient noise, which can help escape local minima.
    #   - May improve generalization by avoiding sharp, overfit solutions.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # Number of steps to accumulate gradients before performing an optimization step.
    # - Useful when VRAM is limited and larger batch sizes don't fit in memory.
    # - Effectively simulates a larger batch by accumulating gradients over multiple smaller batches.
    # - Setting this >1 means fewer optimization steps per epoch, but similar gradient updates as using a larger batch.
    # - **Cons:**
    #   - **Slower updates**: Since weights are updated less frequently, training may take longer.
    #   - **Training instability**: Large accumulated gradients can lead to instability if learning rate is not adjusted.
    #   - **Requires learning rate tuning**: Since it changes the effective batch size, the learning rate should be scaled accordingly.
    #   - **Higher memory usage for gradients**: Even though the batch size per step is small, stored gradients take up memory.
    gradient_accumulation_steps=1,
    # Use BF16 if available:
    # - higher number range than FP16
    # - less precision than FP16
    # - less memory usage than FP16
    fp16=not DEVICE_SUPPORTS_BF16,
    bf16=DEVICE_SUPPORTS_BF16,
    # Perform evaluation every training epoch by
    # calling `compute_metrics` on the `eval_dataset` (see below in `Trainer` instantiation)
    eval_strategy="epoch",
    # Save best model every training epoch
    save_strategy="epoch",
    # Consider best model the one with best validation loss
    # (checkpointing on best accuracy may save a overfitted
    # model that has great accuracy on validation set but not
    # in test set or real world)
    metric_for_best_model="eval_loss",
    # Lower validation loss is better (default is greater is better)
    greater_is_better=False,
    # When training ends load weights of best model found during training
    load_best_model_at_end=True,
    # Disable WandB logging
    report_to="none",
    logging_first_step=True
)

trainer = Trainer(
    model, # The model to be trained
    training_args, # The training settings
    train_dataset=tokenized_dataset["train"], # The dataset to train the model on
    eval_dataset=tokenized_dataset["validation"], # The dataset to evaluate the model on
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)], # Stop if validation loss doesn't improve for 3 epochs
    processing_class=tokenizer, # Tokenizer to pair with specified model
    data_collator=data_collator, # The data collator to run through batch before using it (eg: pad sequences to max sequence length in that batch)
    compute_metrics=compute_metrics  # Function that calculates the evaluation metrics
)
trainer

First, let's evaluate the model 📊 to establish our baseline:

In [ ]:
trainer.evaluate()

Without finetuning, the model is about 50% accurate 🎯, indicating random guesses 🤔. Now, let's train the model 📚:

In [ ]:
trainer.train()

Evaluation accuracy improved significantly in just a few epochs. 📈✨

To push the trained model to the Hugging Face hub, uncomment and run the cell below: 🐻💻✨

In [ ]:
#trainer.push_to_hub()

## Evaluate Model 📊🔍

Running `evaluate()` will assess the dataset specified in `eval_dataset` for the `Trainer` 🏃‍♂️📊:

In [ ]:
trainer.evaluate()

This is equivalent to directly specifying the dataset 📊:

In [ ]:
trainer.evaluate(tokenized_dataset["validation"])

The final evaluation post-training will be on the test dataset 📊, which was not used during training. 🏋️‍♂️

In [ ]:
trainer.evaluate(tokenized_dataset["test"])

Let's dump some misclassified examples. 📊 First, we need the predictions for the test set: 🧪

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])
logits = predictions.predictions
predicted_labels = np.argmax(logits, axis=-1)
predicted_labels

Identify the mislabeled samples: 🕵️‍♂️🔍

In [ ]:
true_labels = predictions.label_ids
misclassified_indices = np.where(predicted_labels != true_labels)[0]
len(misclassified_indices)

Let's create a table of mismatches: 📊✨

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import Markdown

# Sample N texts and labels
label_names = raw_dataset["train"].features["label"].names
sample_indices = np.random.choice(misclassified_indices, size=5, replace=False)
sample_texts = raw_dataset["test"].select(sample_indices)["text"]
sample_true_labels = [label_names[x] for x in np.array(true_labels)[sample_indices].tolist()]
sample_predicted_labels = [label_names[x] for x in np.array(predicted_labels)[sample_indices].tolist()]

# Identify which samples were truncated when tokenized
# (could truncation be affecting classification?)
input_ids = tokenized_dataset["test"].select(sample_indices)["input_ids"]
text_lengths = [len(x) for x in input_ids]
sample_truncated = [x > tokenizer.model_max_length for x in text_lengths]

# Create dataframe
df = pd.DataFrame({
    "Index": sample_indices,
    "Text": sample_texts,
    "True Label": sample_true_labels,
    "Predicted Label": sample_predicted_labels,
    "Truncated" : sample_truncated
})

# Render as markdown
markdown_table = df.to_markdown(index=False)
Markdown(markdown_table)